# Tags Retrieval Benchmark
Measure three scores:
- Jaccard simmularity
- Accuracy
- Callback

In [1]:
# %% [markdown]
# # Tags Retrieval Benchmark
# Run this file directly or as an interactive notebook cell.

# %%
import os
import sys
import pandas as pd

Because this notebook file was run in `benchmark` forlder, it's working directory needs to move to root directory in order to call other utils normally.

In [2]:
INPUT_PATH = 'data/output/SteamGames_Clean_1.csv'
# Adjust Working Directory to Project Root
# Checks if the script/notebook is currently in the 'benchmark' folder
current_dir = os.getcwd()
if current_dir.endswith('benchmark'):
    root_dir = os.path.dirname(current_dir)
    os.chdir(root_dir)
    sys.path.append(root_dir)

print(f"Active Working Directory: {os.getcwd()}")

Active Working Directory: /home/leezy/Coding/mini_project/steam_game


In [ ]:
# import after redirect our working directory to root of the project
from utils.preprocess.imputation.genres_tags.func import RetrieveRequest, retrieve_tag_list

In [14]:
def calculate_jaccard_similarity(true_tags: set, pred_tags: set) -> float:
    """Calculates Jaccard Similarity: Intersection over Union."""
    if not true_tags and not pred_tags:
        return 1.0
    intersection = len(true_tags.intersection(pred_tags))
    union = len(true_tags.union(pred_tags))
    return intersection / union if union > 0 else 0.0

def calculate_recall(true_tags: set, pred_tags: set) -> float:
    """Calculates Recall (Callback): How many of the true tags were retrieved."""
    if not true_tags:
        return 1.0
    intersection = len(true_tags.intersection(pred_tags))
    return intersection / len(true_tags)

def calculate_precision(true_tags: set, pred_tags: set) -> float:
    """Precision: How many predicted tags are actually correct."""
    if not pred_tags:
        return 1.0 if not true_tags else 0.0

    intersection = len(true_tags.intersection(pred_tags))
    return intersection / len(pred_tags)


def calculate_f1(precision: float, recall: float) -> float:
    """F1-score: Harmonic mean of Precision and Recall."""
    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

def calculate_accuracy(true_tags: set, pred_tags: set) -> int:
    """Calculates Exact Match Accuracy: 1 if sets match perfectly, 0 otherwise."""
    return 1 if true_tags == pred_tags else 0



# Pipeline
def evaluate_retrieval(df_eval):
    results = []

    for idx, row in df_eval.iterrows():
        true_set = (
            set(row['True_Tags'])
            if isinstance(row['True_Tags'], (list, set))
            else set()
        )

        pred_set = (
            set(row['Predicted_Tags'])
            if isinstance(row['Predicted_Tags'], (list, set))
            else set()
        )

        precision = calculate_precision(true_set, pred_set)
        recall = calculate_recall(true_set, pred_set)

        results.append({
            'Jaccard_Similarity': calculate_jaccard_similarity(
                true_set, pred_set
            ),
            'Precision': precision,
            'Recall': recall,
            'F1_Score': calculate_f1(precision, recall),
            'Exact_Match': calculate_accuracy(true_set, pred_set)
        })

    metrics_df = pd.DataFrame(results)

    print("--- Retrieval Benchmark Results ---")
    print(f"Mean Jaccard:  {metrics_df['Jaccard_Similarity'].mean():.4f}")
    print(f"Mean Precision: {metrics_df['Precision'].mean():.4f}")
    print(f"Mean Recall:    {metrics_df['Recall'].mean():.4f}")
    print(f"Mean F1-score:  {metrics_df['F1_Score'].mean():.4f}")
    print(f"Exact Match:    {metrics_df['Exact_Match'].mean():.4f}")

    return metrics_df


Run with `n_sample` to check for scores

In [ ]:

n_sample = 500

# Load testing data
df = pd.read_csv(INPUT_PATH)
df = df.sample(frac=1).reset_index(drop=True)
df.dropna(subset=['Name', 'Description', 'Tags'], inplace=True)

df_test = df.head(n_sample)
tags_list = [None] * len(df_test)

for i, (_, row) in enumerate(df_test.iterrows()):
    game_name = row['Name'] if pd.notna(row['Name']) else ""
    desc_text = str(row['Description']) if pd.notna(row['Description']) else ""

    tags = (
        [tag.strip() for tag in row['Tags'].split(',')]
        if pd.notna(row['Tags'])
        else []
    )

    response = retrieve_tag_list(
        RetrieveRequest(
            name=game_name,
            description=desc_text,
            genres=tags,
            top_k=10,
            top_tag=10,
            top_genre=4,
        )
    )

    tags_list[i] = response.tag_list if response.tag_list else None

df_eval = pd.DataFrame({
    'True_Tags': df_test['Tags'].apply(
        lambda x: [tag.strip() for tag in x.split(',')]
    ),
    'Predicted_Tags': tags_list
})

evaluate_retrieval(df_eval)

--- Retrieval Benchmark Results ---
Mean Jaccard:  0.4118
Mean Precision: 0.7195
Mean Recall:    0.5595
Mean F1-score:  0.5658
Exact Match:    0.0160


,Jaccard_Similarity,Precision,Recall,F1_Score,Exact_Match
0,0.363636,0.8,0.400000,0.533333,0
1,0.428571,0.9,0.450000,0.600000,0
2,0.428571,0.9,0.450000,0.600000,0
3,0.526316,1.0,0.526316,0.689655,0
4,0.800000,0.8,1.000000,0.888889,0
...,...,...,...,...,...
1995,0.428571,0.9,0.450000,0.600000,0
1996,0.363636,0.8,0.400000,0.533333,0
1997,0.166667,0.2,0.500000,0.285714,0
1998,0.357143,0.5,0.555556,0.526316,0
